In [1]:
import numpy as np
import math
from datetime import datetime, timedelta
from dash import Dash
from src.data.data_tools import Database, SENSOR_LIST, ddmm_to_decimal, export_imu_readings_to_csv
from src.ekf.ekf_utils import ekf_navigation, ekf_to_coor

app = Dash()

sample_limit = 100
db = Database("./data/dataset.db", calibration_file="./data/calibration.json")

gps_readings = db.get_gps_readings(start_datetime=datetime(2026, 3, 18, 21), end_datetime=datetime(2026, 3, 19), min_satellite_count=4)

# subsample list to defined count starting at a zero velocity sample
for idx, reading in enumerate(gps_readings):
    if reading.speed <= 0.05 and idx + 1 < len(gps_readings) and gps_readings[idx + 1].speed <= 0.05:
        gps_readings = gps_readings[idx:idx+sample_limit]
        break

for idx, reading in enumerate(gps_readings):
    if idx > 0 and reading.timestamp - gps_readings[idx-1].timestamp > timedelta(seconds=1.1):
        gps_readings = gps_readings[0:idx]
        break

imu_readings = db.get_imu_readings(start_datetime=gps_readings[0].timestamp, end_datetime=gps_readings[-1].timestamp)

gps_coor = [(ddmm_to_decimal(gps.latitude), ddmm_to_decimal(gps.longitude)) for gps in gps_readings]
start_lat, start_lon = gps_coor[0]

ekf_q_map = {}
ekf_p_map = {}
ekf_v_map = {}
ekf_coor_map = {}
min_sample = math.inf

export_imu_readings_to_csv(imu_readings, "imu_readings.csv")

for sensor in SENSOR_LIST:

    # decompose sensor readings to separate arrays
    sensor_readings = [r for r in imu_readings if r.sensor_name == sensor]
    gyr = np.array([np.array(r.gyr) for r in sensor_readings])
    acc = np.array([np.array(r.acc) for r in sensor_readings])
    mag = np.array([np.array(r.mag) for r in sensor_readings])
    time = np.array([r.timestamp for r in sensor_readings])

    print(gyr)
    print(f"Running EKF on sensor {sensor} - {len(sensor_readings)} readings")
    print(f"Initial GPS data:"
          f"\n\tposition: {start_lat:.2f}, {start_lon:.2f}"
          f"\n\tcourse: {gps_readings[0].course}"
          f"\n\tsamples: {len(gps_coor)}")
    print(f"Initial IMU data:"
          f"\n\tmag heading: {math.degrees(math.atan2(mag[0, 0], mag[0, 1]))}"
          f"\n\tsamples: {len(sensor_readings)}"
          f"\n\trate: {len(sensor_readings)/len(gps_coor)}")

    Q, V, P = ekf_navigation(gyr, acc, mag, time, mag_ref=[.237617,	.45396,	.383970])

    ekf_coor_map[sensor] = ekf_to_coor(start_lat, start_lon, P)
    min_sample = min(min_sample, len(ekf_coor_map[sensor]))

    ekf_q_map[sensor] = Q
    ekf_p_map[sensor] = P
    ekf_v_map[sensor] = V

stacked = np.array([ekf_p_map[s][:min_sample] for s in SENSOR_LIST])  # shape (6, min_sample, 3)
avg_ekf = stacked.mean(axis=0)

# ekf_coor_map["avg"] = ekf_to_coor(start_lat, start_lon, avg_ekf)

Successfully exported 62604 records to 'imu_readings.csv'.
[[ 0.00851878 -0.00847357  0.00131578]
 [ 0.00962324 -0.00814832  0.00228507]
 [ 0.01059633 -0.01420609  0.00087208]
 ...
 [ 0.0030145  -0.01953201 -0.00155467]
 [ 0.00741432 -0.00879882  0.00034649]
 [ 0.00208412 -0.00674368 -0.00011189]]
Running EKF on sensor icm_20948_1 - 10434 readings
Initial GPS data:
	position: 34.16, -118.46
	course: 290.54
	samples: 100
Initial IMU data:
	mag heading: 56.690084919983086
	samples: 10434
	rate: 104.34
Bias = [ 0.32355698 -0.14960727  9.81899956]
[[-0.04788239  0.00507028  0.00182979]
 [-0.04786485  0.00372022  0.00788658]
 [-0.04360797  0.00371815  0.00787379]
 ...
 [-0.05212648  0.00641589  0.00588188]
 [-0.04680219  0.00674919  0.00687572]
 [-0.05320047 -0.00738098  0.00486659]]
Running EKF on sensor icm_20948_2 - 10433 readings
Initial GPS data:
	position: 34.16, -118.46
	course: 290.54
	samples: 100
Initial IMU data:
	mag heading: 32.640024846985305
	samples: 10433
	rate: 104.33
Bias

In [5]:
"""
Plot position data
"""

from src.data.plotting import gps_scatter_map, imu_scatter_map
import pandas as pd
import plotly.graph_objects as go
from dash import Dash, html, dcc
import numpy as np


map_fig = go.Figure()

gps_scatter_map(map_fig, gps_readings)

# Add EKF Traces to Map
for sensor in ekf_coor_map.keys():
    imu_scatter_map(
        map_fig,
        gps_readings[0].latitude,
        gps_readings[0].longitude,
        sensor,
        [imu.timestamp for imu in imu_readings if imu.sensor_name == sensor],
        ekf_q_map[sensor],
        ekf_v_map[sensor],
        ekf_p_map[sensor]
    )

gps_lats, gps_lons, gps_times = zip(*[(ddmm_to_decimal(gps.latitude), ddmm_to_decimal(gps.longitude), (gps.timestamp - gps_readings[0].timestamp).total_seconds()) for gps in gps_readings])

center_lat = np.mean(gps_lats)
center_lon = np.mean(gps_lons)

# Center map on GPS data
map_fig.update_layout(
    map=dict(style='open-street-map', center=dict(lat=center_lat, lon=center_lon), zoom=15),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=0, r=0, t=40, b=0),
)


# 3. DASH APP LAYOUT
map_app = Dash(__name__)

map_app.layout = html.Div([
    html.Div([
        dcc.Graph(figure=map_fig)
    ], style={'padding': '0px'}),
])

map_app.run()

In [3]:
"""
Plot velocity
"""



# 2. CREATE THE VELOCITY LINE CHART FIGURE (The new part)
vel_fig = go.Figure()

axis_colors = {
    'x': 'royalblue',
    'y': 'crimson',
    'z': 'forestgreen'
}

for sensor, velocities in ekf_v_map.items():
    # We use an index as the X-axis (Time/Sample count)
    # If you have a timestamp array for each sensor, replace np.arange with that.
    v_arr = np.array(velocities)
    time_axis = [imu.timestamp for imu in imu_readings if
                 gps_readings[0].timestamp <= imu.timestamp <= gps_readings[-1].timestamp and (
                             imu.sensor_name == sensor)]
    # time_axis = np.arange(len(v_arr))
    # Extract components
    vx = v_arr[:, 0]
    vy = v_arr[:, 1]
    vz = v_arr[:, 2]

    # Add Trace for X-component
    vel_fig.add_trace(go.Scatter(
        x=time_axis, y=vx,
        mode='lines',
        name=f'{sensor} (N)',
        line=dict(color=axis_colors['x'], width=1.5),
        opacity=0.8  # Slightly transparent to handle overlapping lines
    ))

    # Add Trace for Y-component
    vel_fig_trace_y = vel_fig.add_trace(go.Scatter(
        x=time_axis, y=vy,
        mode='lines',
        name=f'{sensor} (E)',
        line=dict(color=axis_colors['y'], width=1.5),
        opacity=0.8
    ))

    # Add Trace for Z-component
    vel_fig.add_trace(go.Scatter(
        x=time_axis, y=vz,
        mode='lines',
        name=f'{sensor} (D)',
        line=dict(color=axis_colors['z'], width=1.5),
        opacity=0.8
    ))



vn = [gps.speed * np.cos(np.radians(gps.course) )for gps in gps_readings]
ne = [gps.speed * np.sin(np.radians(gps.course) )for gps in gps_readings]
time_axis = [gps.timestamp for gps in gps_readings]


vel_fig.add_trace(go.Scatter(
    x=time_axis, y=vn,
    mode='lines',
    name='GPS (N)',
    line=dict(color="purple", width=1.5),
    opacity=0.8
))

vel_fig.add_trace(go.Scatter(
    x=time_axis, y=ne,
    mode='lines',
    name='GPS (E)',
    line=dict(color="goldenrod", width=1.5),
    opacity=0.8
))

vel_fig.update_layout(
    title='Sensor Velocity Over Time',
    xaxis_title='Sample Index (Time)',
    yaxis_title='Velocity (m/s)',
    legend=dict(x=0.01, y=0.99),
    template='plotly_white',
    margin=dict(l=40, r=20, t=50, b=40)
)


# 3. DASH APP LAYOUT
vel_app = Dash(__name__)

vel_app.layout = html.Div([
    # Container for the Velocity Chart
    html.Div([
        dcc.Graph(figure=vel_fig)
    ], style={'padding': '0px'})
])

vel_app.run()

In [4]:
import numpy as np
from dash import Dash, html, dcc, Input, Output, State
import plotly.graph_objects as go
import plotly.colors as pc
from scipy.spatial.transform import Rotation as R_func

# --- Configuration & Data Validation ---
APP_NAME = "Multi-Sensor EKF Workspace"
SAMPLING_STEP = 20
ORIENTATION_SCALE_ADJUSTMENT = 0.5

try:
    if 'ekf_p_map' not in globals() or 'ekf_q_map' not in globals():
        raise NameError("Required maps 'ekf_p_map' or 'ekf_q_map' not found.")

    sensor_names = sorted(list(ekf_p_map.keys()))
    common_min_len = min(len(ps) for ps in ekf_p_map.values())
    colors = pc.qualitative.Plotly[:len(sensor_names)]
    sensor_color_map = {name: colors[i] for i, name in enumerate(sensor_names)}

    all_p_coords = np.concatenate([ekf_p_map[s] for s in sensor_names])
    world_min, world_max = all_p_coords.min(axis=0), all_p_coords.max(axis=0)
    world_range = world_max - world_min
    DYNAMIC_AXIS_LEN = (np.max(world_range) * 0.15) * ORIENTATION_SCALE_ADJUSTMENT

    # --- Dash Application Setup ---
    app = Dash(__name__)

    # Create the slider component outside of the layout definition to avoid syntax errors
    time_slider_component = dcc.Slider(
        id='time-slider',
        min=0,
        max=common_min_len - 1,
        step=1,
        value=0,
        marks={i: f"Idx {i}" for i in range(0, common_min_len, max(1, common_min_len//20))},
        tooltip={"always_visible": True}
    )

    app.layout = html.Div([
        dcc.Store(id='camera-store'),
        dcc.Graph(id='multi-sensor-graph', style={'height': '75vh'}),
        html.Div([
            html.P("Global Timeline Index:", style={'textAlign': 'center', 'fontWeight': 'bold'}),
            time_slider_component # Now correctly placed as a child
        ], style={'padding': '30px', 'backgroundColor': '#ecf0f1', 'borderRadius': '15px', 'margin': '20px'})
    ], style={'backgroundColor': '#ffffff', 'margin': '0 auto', 'width': '95%'})

    # --- Callback 1: Sync Browser Camera -> Python Store ---
    @app.callback(
        Output('camera-store', 'data'),
        Input('multi-sensor-graph', 'relayoutData')
    )
    def update_camera_memory(relayout_data):
        if not relayout_data:
            return None
        for key, val in relayout_data.items():
            if 'camera' in key:
                return val
        return None

    # --- Callback 2: Render Graph (Using memory) ---
    @app.callback(
        Output('multi-sensor-graph', 'figure'),
        Input('time-slider', 'value'),
        State('camera-store', 'data')
    )
    def render_graph(current_idx, saved_camera):
        fig = go.Figure()

        for sensor in sensor_names:
            p_arr, q_arr = ekf_p_map[sensor], ekf_q_map[sensor]
            color = sensor_color_map[sensor]
            path_idx = np.arange(0, current_idx + 1, SAMPLING_STEP)

            # Trace: Path
            fig.add_trace(go.Scatter3d(
                x=p_arr[path_idx, 0], y=p_arr[path_idx, 1], z=p_arr[path_idx, 2],
                mode='lines', line=dict(color=color, width=3), name=f'{sensor} Path'
            ))

            # Trace: Position & Axes
            curr_p = p_arr[current_idx]
            fig.add_trace(go.Scatter3d(
                x=[curr_p[0]], y=[curr_p[1]], z=[curr_p[2]],
                mode='markers', marker=dict(color=color, size=6), name=f'{sensor} Pos'
            ))

            # Trace: Orientation Axes (Dynamic Scaling)
            try:
                rot = R_func.from_quat(q_arr[current_idx], scalar_first=True).as_matrix()
                for i, d in enumerate([[1, 0, 0], [0, 1, 0], [0, 0, 1]]):
                    end = curr_p + rot @ (np.array(d) * DYNAMIC_AXIS_LEN)
                    fig.add_trace(go.Scatter3d(
                        x=[curr_p[0], end[0]], y=[curr_p[1], end[1]], z=[curr_p[2], end[2]],
                        mode='lines', line=dict(color=['red','green','blue'][i], width=6),
                        showlegend=False
                    ))
            except: pass

        fig.update_layout(
            scene=dict(xaxis_title='N', yaxis_title='E', zaxis_title='D',
                       aspectmode='data', camera=saved_camera),
            showlegend=True, transition_duration=150
        )
        return fig

    app.run(mode='inline', port=8051)

except Exception as e:
    print(f"Error: {e}")